# Exercise 5.4
After saving the weights, load the model and optimizer in a new Python session or
Jupyter notebook file and continue pretraining it for one more epoch using the
train_model_simple function.

In [ ]:
import sys
sys.path.append('../01_main-chapter-code')

In [ ]:
import torch
import tiktoken

from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}


# load data

In [ ]:
pwd

In [ ]:
from previous_chapters import create_dataloader_v1
# Alternatively:
# from llms_from_scratch.ch02 import create_dataloader_v1

with open("../01_main-chapter-code/the-verdict.txt", "r") as f:
    text_data = f.read()
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
from gpt_train import train_model_simple

In [ ]:
model = GPTModel(GPT_CONFIG_124M)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)



In [ ]:
checkpoint_path = "../01_main-chapter-code/model_and_optimizer.pth"
checkpoint = torch.load(checkpoint_path, weights_only=True, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
#model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))


optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")



In [ ]:
model.to(device)

# train for one more epoch

In [ ]:
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=1, eval_freq=5, eval_iter=1,
    start_context="Every effort moves you", tokenizer=tokenizer
    )